# 02 — Feature Engineering

A feature is an input the model can use. We combine flight schedules with earlier weather and delay history.

**A** uses schedule information. **B** adds weather, earlier delays and airport traffic.
To preserve the existing scores, we keep the original feature formulas and the same 50,000 flights.

This is the most involved notebook because the time joins must exclude future information.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

Find the project folder. Starting in the notebook folder, move up until `config/project.json` exists.

In [2]:
ROOT = Path.cwd()
while not (ROOT / 'config/project.json').is_file():
    if ROOT == ROOT.parent:
        raise FileNotFoundError('Open this notebook inside the project folder.')
    ROOT = ROOT.parent
OUT = ROOT / 'reports/simple_reproduction'
OUT.mkdir(parents=True, exist_ok=True)

In [3]:
import re
from sklearn.model_selection import train_test_split
selection = json.loads((ROOT/'reports/final_models/selection.json').read_text())
cohorts = json.loads((ROOT/'reports/final_models/cohorts.json').read_text())
flights = pd.read_parquet(ROOT/'data/processed/flights_clean.parquet')

## 1. Select the same training and test flights

Keep the saved 40,000 training IDs. Recreate the July sample with seed 42 and the same delayed/on-time proportions.

In [4]:
in_july = flights['flight_date'].between('2026-07-01', '2026-07-31')
july = flights.loc[in_july & flights['label_eligible']].copy()
july = july.sort_values('flight_id')
strata = july['month'].astype(str) + '|' + july['delayed_departure'].astype(str)
chosen, _ = train_test_split(july.index, train_size=10000, stratify=strata, random_state=42)
selected_july = july.loc[chosen].sort_values('flight_id')
july_ids = selected_july['flight_id'].tolist()

In [5]:
training_ids = cohorts['final_refit_ids']
selected_ids = training_ids + july_ids
indexed_flights = flights.set_index('flight_id', drop=False)
features = indexed_flights.loc[selected_ids].reset_index(drop=True)
is_training = features['flight_id'].isin(training_ids)
features['split'] = np.where(is_training, 'train', 'test')

In [6]:
split_counts = features.groupby('split').size()
assert split_counts.to_dict() == {'test': 10000, 'train': 40000}
split_dates = features.groupby('split')['flight_date'].agg(['size', 'min', 'max'])
display(split_dates)

,size,min,max
split,,,
test,10000,2026-07-01,2026-07-31
train,40000,2025-01-01,2025-12-31


## 2. Calendar features

Sine and cosine represent time on a circle: 23:59 and 00:01 should be close together.
The second pair allows two peaks per day/year. Airport-specific copies let patterns differ across airports.

In [7]:
day_angle = 2 * np.pi * features['scheduled_departure_minutes'] / 1440
year_angle = 2 * np.pi * (features['month'] - 1) / 12
features['departure_sin2'] = np.sin(2 * day_angle)
features['departure_cos2'] = np.cos(2 * day_angle)
features['month_sin2'] = np.sin(2 * year_angle)
features['month_cos2'] = np.cos(2 * year_angle)

For example, `atl_departure_sin` equals the time feature for ATL flights and zero for other airports.

In [8]:
for origin in ['ATL', 'ORD', 'DFW']:
    at_airport = (features['origin'] == origin).astype(float)
    for harmonic in ['sin', 'cos', 'sin2', 'cos2']:
        source_column = 'departure_' + harmonic
        new_column = origin.lower() + '_' + source_column
        features[new_column] = at_airport * features[source_column]

## 3. Read and clean weather reports

Read the original API CSV files. Remove identical reports repeated across downloads.
If a station/time has conflicting reports, exclude all versions rather than choosing one arbitrarily.

In [9]:
WEATHER_COLUMNS = ['station','valid','tmpf','dwpf','drct','sknt','gust','vsby','p01i',
    'skyc1','skyc2','skyc3','skyc4','skyl1','skyl2','skyl3','skyl4','wxcodes','metar']

In [10]:
WEATHER_BASE = ['temperature_f','wind_knots','gust_knots','visibility_miles','precip_inches',
    'precip_trace','ceiling_ft','no_ceiling_reported','cloud_coverage_missing','rain_reported',
    'frozen_precip_reported','fog_or_mist_reported','thunder_reported',
    'condition_codes_missing','ceiling_height_missing']

In [11]:
WEATHER_NEW = ['dewpoint_spread_f','wind_east_knots','wind_north_knots','freezing_wet',
    'prior3h_report_n','prior3h_precip_known_n','prior3h_wet_fraction',
    'prior3h_visibility_min','prior3h_gust_max','prior3h_visibility_change']

In [12]:
def ns_utc(series):
    return pd.to_datetime(series, utc=True).astype('datetime64[ns, UTC]')

In [13]:
weather_folder = ROOT / 'data/raw/weather/observations'
paths = sorted((weather_folder / '2025').glob('*.csv'))
paths += sorted((weather_folder / '2026').glob('2026_0[5-7]_origins.csv'))
parts = []
for path in paths:
    raw = pd.read_csv(path, usecols=WEATHER_COLUMNS, dtype='string', keep_default_na=False)
    keep = raw['station'].str.strip().isin(['ATL', 'ORD', 'DFW'])
    raw = raw.loc[keep].copy()
    raw['weather_source_file'] = path.relative_to(ROOT).as_posix()
    parts.append(raw)

In [14]:
weather = pd.concat(parts, ignore_index=True)
for column in WEATHER_COLUMNS:
    weather[column] = weather[column].str.strip()
weather = weather.drop_duplicates(WEATHER_COLUMNS).copy()

In [15]:
weather['weather_observed_utc'] = ns_utc(weather['valid'])
report_key = ['station', 'weather_observed_utc']
conflict = weather.duplicated(report_key, keep=False)
conflict_count = weather.loc[conflict, report_key].drop_duplicates().shape[0]
weather = weather.loc[~conflict].copy()
weather = weather.rename(columns={'station': 'origin'})
print('Conflicting station/times excluded:', conflict_count)

Conflicting station/times excluded: 7


### Numeric measurements

`M`/blank means unknown. Trace precipitation becomes numeric zero plus a trace flag. Negative wind, visibility, precipitation and cloud heights become missing; negative temperature remains valid. Valid statistical extremes are not capped.

In [16]:
weather['precip_trace'] = weather['p01i'].eq('T').astype(float)
numeric_weather = ['tmpf', 'dwpf', 'drct', 'sknt', 'gust', 'vsby', 'p01i',
                   'skyl1', 'skyl2', 'skyl3', 'skyl4']
nonnegative_weather = ['sknt', 'gust', 'vsby', 'p01i', 'skyl1', 'skyl2', 'skyl3', 'skyl4']

First save which precipitation reports said “trace”. Then convert the numeric columns; unknown codes become missing.

In [17]:
trace_reported = weather['p01i'] == 'T'
for column in numeric_weather:
    raw_values = weather[column].replace(['M', 'T', ''], pd.NA)
    numeric_values = pd.to_numeric(raw_values, errors='coerce').astype(float)
    weather[column] = numeric_values.where(np.isfinite(numeric_values))
weather.loc[trace_reported, 'p01i'] = 0.0

Remove physically invalid negative measurements. A negative temperature is valid and stays in the data.

In [18]:
for column in nonnegative_weather:
    negative = weather[column] < 0
    weather[column] = weather[column].mask(negative)
known_precipitation = weather['p01i'].notna()
weather['precip_trace'] = weather['precip_trace'].where(known_precipitation)

### Cloud height

A ceiling is the lowest broken, overcast or vertically obscured cloud layer.
First inspect the structured cloud columns. Unknown heights stay missing.

In [19]:
coverage_columns = ['skyc1', 'skyc2', 'skyc3', 'skyc4']
height_columns = ['skyl1', 'skyl2', 'skyl3', 'skyl4']
coverage = weather[coverage_columns]
heights = weather[height_columns].copy()
heights.columns = coverage_columns
ceiling = coverage.isin(['BKN', 'OVC', 'VV'])

In [20]:
known_cloud_codes = ['CLR', 'SKC', 'FEW', 'SCT', 'BKN', 'OVC', 'VV', 'NSC', 'NCD']
coverage_known = coverage.isin(known_cloud_codes).any(axis=1)
unknown_ceiling_height = ceiling & heights.isna()
ceiling_height_missing = unknown_ceiling_height.any(axis=1)

The raw METAR text may contain extra layers. Read the main report before its `RMK` remarks.
In these codes, `BKN020` means broken cloud at 2,000 feet; `///` means unknown height.

The pattern recognises cloud codes such as `BKN020` or `OVC///`. The surrounding boundary checks prevent matching inside other words.

In [21]:
body = weather['metar'].str.split(r'\s+RMK(?:\s|$)', n=1, regex=True).str[0]
cloud_code = r'(?:FEW|SCT|BKN|OVC|VV)'
cloud_height = r'(?:\d{3}|///)'
cloud_suffix = r'(?:CB|TCU)?'
cloud_pattern = r'(?<!\S)' + cloud_code + cloud_height + cloud_suffix + r'(?=\s|=|$)'
cloud_groups = body.str.findall(cloud_pattern)

`explode()` puts each cloud layer on its own row while keeping the original weather-row index. Keep only layers that form a ceiling.

In [22]:
layers = cloud_groups.explode().dropna()
is_ceiling_layer = layers.str.startswith(('BKN', 'OVC', 'VV'))
ceiling_layers = layers.loc[is_ceiling_layer]

Extract the height and group the layers back into one result per weather report. Three digits represent hundreds of feet.

In [23]:
height_text = ceiling_layers.str.extract(r'(\d{3})', expand=False)
layer_heights = pd.to_numeric(height_text).astype(float) * 100
body_ceiling = layer_heights.groupby(level=0).min()
body_has_ceiling = ceiling_layers.groupby(level=0).size() > 0
unknown_height = ceiling_layers.str.contains('///', regex=False)
body_height_missing = unknown_height.groupby(level=0).any()

In [24]:
body_ceiling = body_ceiling.reindex(weather.index)
body_has_ceiling = body_has_ceiling.reindex(weather.index, fill_value=False)
body_height_missing = body_height_missing.reindex(weather.index, fill_value=False)

Combine the text report with the structured columns. An explicitly unknown ceiling height stays missing.

In [25]:
body_has_clouds = cloud_groups.str.len() > 0
body_is_clear = body.str.contains(r'\b(?:CLR|SKC|NSC|NCD)\b')
coverage_known = coverage_known | body_has_clouds | body_is_clear
ceiling_height_missing = ceiling_height_missing | body_height_missing
column_ceiling = heights.where(ceiling).min(axis=1)
ceiling_sources = pd.concat([column_ceiling, body_ceiling], axis=1)
weather['ceiling_ft'] = ceiling_sources.min(axis=1).mask(ceiling_height_missing)
has_ceiling = ceiling.any(axis=1) | body_has_ceiling

In [26]:
weather['no_ceiling_reported'] = (~has_ceiling).astype(float).where(coverage_known)
weather['cloud_coverage_missing'] = (~coverage_known).astype(float)
weather['ceiling_height_missing'] = ceiling_height_missing.astype(float).where(coverage_known)

### Weather conditions

Convert reported rain, snow, fog and thunder into flags. Missing codes mean unknown, not definitely dry.

In [27]:
conditions = weather['wxcodes']
condition_known = ~conditions.isin(['M', ''])
condition_patterns = {'rain_reported': 'RA|DZ', 'frozen_precip_reported': 'SN|SG|PL|IC|GS|GR',
                      'fog_or_mist_reported': 'FG|BR', 'thunder_reported': 'TS'}
for name, pattern in condition_patterns.items():
    reported = conditions.str.contains(pattern, regex=True, na=False)
    weather[name] = reported.astype(float).where(condition_known)
weather['condition_codes_missing'] = (~condition_known).astype(float)

Rename the source abbreviations into readable names. `tmpf` means temperature in Fahrenheit; `sknt` means wind speed in knots.

In [28]:
weather_names = {'tmpf': 'temperature_f', 'sknt': 'wind_knots', 'gust': 'gust_knots',
                 'vsby': 'visibility_miles', 'p01i': 'precip_inches'}
weather = weather.rename(columns=weather_names)

### Wind and cold/wet conditions

Derive temperature minus dewpoint, wind direction components, and a freezing-and-wet flag.

In [29]:
valid_direction = weather['drct'].between(0, 360)
direction = weather['drct'].where(valid_direction)
radians = np.deg2rad(direction)
calm = weather['wind_knots'] == 0
weather['dewpoint_spread_f'] = weather['temperature_f'] - weather['dwpf']
weather['wind_east_knots'] = weather['wind_knots'] * np.sin(radians)
weather['wind_north_knots'] = weather['wind_knots'] * np.cos(radians)
weather.loc[calm, ['wind_east_knots', 'wind_north_knots']] = 0.0

A wet report has measured precipitation, trace precipitation, rain or frozen precipitation. Unknown reports are kept unknown.

In [30]:
measured_precipitation = weather['precip_inches'] > 0
trace_precipitation = weather['precip_trace'] == 1
rain = weather['rain_reported'] == 1
frozen_precipitation = weather['frozen_precip_reported'] == 1
wet = measured_precipitation | trace_precipitation | rain | frozen_precipitation
wet_known = weather['precip_inches'].notna() | weather['rain_reported'].notna()
freezing = weather['temperature_f'] <= 32
known_conditions = wet_known & weather['temperature_f'].notna()
weather['freezing_wet'] = (freezing & wet).astype(float).where(known_conditions)

### Weather during the previous three hours

For each airport, summarise recent reports with rolling windows.
Do not sum precipitation: reported measurement intervals can overlap.

This small helper returns the first non-missing measurement in a window. We use it for both visibility and pressure changes.

In [31]:
def first_known(values):
    known_values = values[np.isfinite(values)]
    if len(known_values) == 0:
        return np.nan
    return known_values[0]

In [32]:
def recent_weather(block):
    block = block.sort_values('weather_observed_utc').set_index('weather_observed_utc')
    wet = (block['precip_inches'] > 0).astype(float)
    wet = wet.where(block['precip_inches'].notna())
    report_count = pd.Series(1., index=block.index)
    roll = block.rolling('3h', closed='both')
    block['prior3h_report_n'] = report_count.rolling('3h', closed='both').sum()
    block['prior3h_precip_known_n'] = roll['precip_inches'].count()
    block['prior3h_wet_fraction'] = wet.rolling('3h', closed='both').mean()
    return block, roll

In [33]:
blocks = []
for origin, block in weather.groupby('origin', observed=True):
    block, roll = recent_weather(block)
    block['prior3h_visibility_min'] = roll['visibility_miles'].min()
    block['prior3h_gust_max'] = roll['gust_knots'].max()
    first_visibility = roll['visibility_miles'].apply(first_known, raw=True)
    block['prior3h_visibility_change'] = block['visibility_miles'] - first_visibility
    blocks.append(block.reset_index())
weather = pd.concat(blocks, ignore_index=True)

## 4. Flight–weather join: backward as-of

For each flight, choose the latest report no later than cutoff minus 20 minutes. The 160-minute tolerance relative to that shifted lookup gives a maximum age of 180 minutes at cutoff. Keep unmatched flights with missing weather. This is many flights to one report, with no multiplication of flight rows.

Prepare the two tables. Save the flight order before sorting by time.

In [34]:
features['_order'] = np.arange(len(features))
cutoff = ns_utc(features['prediction_cutoff_utc'])
features['weather_lookup_utc'] = cutoff - pd.Timedelta(minutes=20)
weather_columns = ['origin', 'weather_observed_utc'] + WEATHER_NEW + WEATHER_BASE
weather_columns += ['weather_source_file']
features['origin'] = features['origin'].astype('string')
right = weather[weather_columns].copy()
right['origin'] = right['origin'].astype('string')

`merge_asof` means “match the nearest eligible time”. `backward` selects an earlier report from the same airport.

In [35]:
left = features.sort_values('weather_lookup_utc')
right = right.sort_values('weather_observed_utc')
features = pd.merge_asof(
    left, right, by='origin',
    left_on='weather_lookup_utc', right_on='weather_observed_utc',
    direction='backward', tolerance=pd.Timedelta(minutes=160),
)

Restore the flight order and calculate how old the matched report was at prediction time.

In [36]:
features = features.sort_values('_order').drop(columns='_order')
features = features.reset_index(drop=True)
cutoff = ns_utc(features['prediction_cutoff_utc'])
report_age = cutoff - features['weather_observed_utc']
features['weather_age_minutes'] = report_age.dt.total_seconds() / 60

In [37]:
features['weather_report_missing'] = features['weather_observed_utc'].isna().astype('int8')
assert len(features) == 50000 and features['flight_id'].is_unique
assert features['weather_age_minutes'].dropna().between(20, 180).all()
display(features[['flight_id', 'prediction_cutoff_utc', 'weather_observed_utc', 'weather_age_minutes']].head())

,flight_id,prediction_cutoff_utc,weather_observed_utc,weather_age_minutes
0,2025_01:100373,2025-01-12 13:38:00+00:00,2025-01-12 12:52:00+00:00,46.0
1,2025_01:100613,2025-01-12 09:45:00+00:00,2025-01-12 08:52:00+00:00,53.0
2,2025_01:10092,2025-01-13 19:06:00+00:00,2025-01-13 17:53:00+00:00,73.0
3,2025_01:10099,2025-01-20 19:06:00+00:00,2025-01-20 17:53:00+00:00,73.0
4,2025_01:10266,2025-01-02 11:10:00+00:00,2025-01-02 09:53:00+00:00,77.0


## 5. Earlier delay rates: count, match, subtract

For each flight, use earlier outcomes from **35 to 7 days before its prediction cutoff**.
Do this separately for each airport, airline/airport, route and flight number.

The calculation is the same for both flight counts and delay counts:

**Count in the window = cumulative count before the end − cumulative count before the start.**

For example, if 120 flights occurred before the end and 100 before the start, the window contains 20 flights.
We match cumulative totals with `merge_asof`, using the same kind of time join as the weather section.

Step 1: sort the known outcomes and build running totals within each group. `cumcount()` counts rows; `cumsum()` adds the 0/1 delay labels.

In [38]:
def running_history(events, group_columns):
    columns = group_columns + ['departure_outcome_event_utc', 'delayed_departure']
    history = events[columns].sort_values('departure_outcome_event_utc').copy()
    groups = history.groupby(group_columns, observed=True, dropna=False)
    history['flights_seen'] = groups.cumcount() + 1
    history['delays_seen'] = groups['delayed_departure'].cumsum()
    return history.drop(columns='delayed_departure')

Step 2: find the last running total strictly before a chosen boundary. Keeping exact matches out includes the window start and excludes the end after subtraction.

In [39]:
def history_before(queries, history, group_columns, boundary):
    left = queries.sort_values(boundary)
    matched = pd.merge_asof(
        left, history, by=group_columns,
        left_on=boundary, right_on='departure_outcome_event_utc',
        direction='backward', allow_exact_matches=False,
    )
    matched = matched.set_index('_query_row').sort_index()
    matched[['flights_seen', 'delays_seen']] = matched[['flights_seen', 'delays_seen']].fillna(0)
    return matched

Step 3: prepare the two boundaries and subtract their totals. The latest contributing event is retained so its timing can be checked.

In [40]:
def history_counts(query, events, group_columns):
    queries = query[group_columns].copy()
    queries['_query_row'] = np.arange(len(query))
    queries['end'] = query['prediction_cutoff_utc'] - pd.Timedelta(days=7)
    queries['start'] = queries['end'] - pd.Timedelta(days=28)
    history = running_history(events, group_columns)
    before_start = history_before(queries, history, group_columns, 'start')
    before_end = history_before(queries, history, group_columns, 'end')
    return subtract_history(before_start, before_end)

In [41]:
def subtract_history(before_start, before_end):
    counts = before_end['flights_seen'] - before_start['flights_seen']
    delays = before_end['delays_seen'] - before_start['delays_seen']
    latest = before_end['departure_outcome_event_utc'].where(counts > 0)
    return counts.astype('int32'), delays, latest

Use the delay count divided by the flight count. For small airline/route/flight groups, blend in the airport rate
with a weight equivalent to 20 flights. Keep the counts so a rate based on few flights is identifiable.

In [42]:
events = flights.loc[flights['label_eligible']]
groups = [
    ('history_origin', ['origin']),
    ('history_airline_origin', ['origin', 'airline']),
    ('history_route', ['origin', 'destination']),
    ('history_flight', ['origin', 'airline', 'flight_number']),
]

Save four fields for each history level: its delay rate, flight count, missing-history flag, and latest outcome time.

In [43]:
def save_history(prefix, counts, rate, latest):
    features[prefix + '_rate'] = rate.to_numpy(dtype=float, na_value=np.nan)
    features[prefix + '_n'] = counts.to_numpy()
    features[prefix + '_no_history'] = (counts == 0).astype('int8').to_numpy()
    features[prefix + '_latest_event_utc'] = latest

In [44]:
for prefix, columns in groups:
    counts, delays, latest = history_counts(features, events, columns)
    if prefix == 'history_origin':
        rate = delays / counts.replace(0, np.nan)
    else:
        airport_rate = features['history_origin_rate']
        rate = (delays + 20 * airport_rate) / (counts + 20)
    save_history(prefix, counts, rate, latest)

Inspect the result: a rate of `0.40` means 40% of the earlier flights in that group were delayed. The counts show how much history supports each rate.

In [45]:
history_columns = ['origin', 'history_origin_n', 'history_origin_rate',
                   'history_airline_origin_n', 'history_airline_origin_rate']
display(features[history_columns].head().round(3))

,origin,history_origin_n,history_origin_rate,history_airline_origin_n,history_airline_origin_rate
0,ATL,25288,0.235,174,0.308
1,ATL,25315,0.235,175,0.316
2,DFW,22707,0.288,11919,0.343
3,DFW,21638,0.322,11294,0.374
4,DFW,22475,0.239,11794,0.293


## 6. Scheduled traffic aggregation

Count all schedules at the origin in **[departure−60 minutes, departure)**. The query itself is excluded. Later cancellations remain in the published schedule context. Availability of the historical schedule at prediction time is an assumption.

In the sorted schedule, `searchsorted` tells us how many flights are before a time. Subtract the counts at the two boundaries to get traffic within the hour.

In [46]:
counts = pd.Series(0, index=features.index)
for origin, query in features.groupby('origin', observed=True):
    airport_rows = flights['origin'] == origin
    schedule = flights.loc[airport_rows, 'scheduled_departure_utc'].sort_values()
    window_end = query['scheduled_departure_utc']
    window_start = window_end - pd.Timedelta(hours=1)
    before_start = schedule.searchsorted(window_start, side='left')
    before_end = schedule.searchsorted(window_end, side='left')
    counts.loc[query.index] = before_end - before_start
features['airport_scheduled_departures_prior_60m'] = counts.astype('int32')

## 7. Pressure and gust features

Read pressure from the matched METAR: `A2992` means 29.92 inches of mercury.
Also record variable wind and how much a gust exceeds the steady wind speed.

In [47]:
PRESSURE = ['altimeter_inhg', 'altimeter_change3h_inhg', 'altimeter_range3h_inhg',
            'variable_wind_flag', 'gust_excess_knots', 'altimeter_missing_flag']

In [48]:
body = weather['metar'].str.split(r'\s+RMK(?:\s|$)', n=1, regex=True).str[0]
pressure_pattern = r'(?<!\S)A(\d{4})(?=\s|=|$)'
pressure_codes = body.str.findall(pressure_pattern)
pressure = pd.to_numeric(pressure_codes.str[0], errors='coerce') / 100
one_pressure_code = pressure_codes.str.len() == 1
weather['altimeter_inhg'] = pressure.where(one_pressure_code).astype(float)

A wind code beginning with `VRB` reports variable direction. Gust excess is gust speed minus steady wind, with a minimum of zero.

In [49]:
wind_pattern = r'(?<!\S)((?:\d{3}|VRB)\d{2,3}(?:G\d{2,3})?KT)(?=\s|=|$)'
wind_code = body.str.extract(wind_pattern, expand=False)
variable_wind = wind_code.str.startswith('VRB').astype(float)
weather['variable_wind_flag'] = variable_wind.where(wind_code.notna())
gust_excess = weather['gust_knots'] - weather['wind_knots']
weather['gust_excess_knots'] = gust_excess.clip(lower=0)

Within each airport, pressure change is the current pressure minus the first known reading in the previous three hours. Range is maximum minus minimum.

In [50]:
blocks = []
for origin, block in weather.groupby('origin', observed=True):
    block = block.sort_values('weather_observed_utc').set_index('weather_observed_utc')
    rolling = block['altimeter_inhg'].rolling('3h', closed='both')
    first_pressure = rolling.apply(first_known, raw=True)
    block['altimeter_change3h_inhg'] = block['altimeter_inhg'] - first_pressure
    block['altimeter_range3h_inhg'] = rolling.max() - rolling.min()
    blocks.append(block.reset_index())
reports = pd.concat(blocks, ignore_index=True)

Attach pressure to the **same weather report already matched to each flight**.
`validate='many_to_one'` checks that the right-hand join keys are unique.

In [51]:
join_keys = ['origin', 'weather_observed_utc']
pressure_columns = PRESSURE[:-1]
left = features[join_keys]
right = reports[join_keys + pressure_columns]
matched = left.merge(right, on=join_keys, how='left', validate='many_to_one')
assert len(matched) == len(features)

In [52]:
for column in pressure_columns:
    features[column] = matched[column].to_numpy(dtype=float, na_value=np.nan)
missing_pressure = features['altimeter_inhg'].isna()
features['altimeter_missing_flag'] = missing_pressure.astype('int8')

### Six transformations used by the existing models

Squares allow curved relationships. Logs reduce the influence of very large positive values.
The inverse visibility feature increases when visibility is poor.

In [53]:
features['temperature_centered_square'] = ((features['temperature_f']-32)/10)**2
features['wind_square'] = features['wind_knots']**2
features['dewpoint_spread_square'] = features['dewpoint_spread_f']**2
features['precip_log1p'] = np.log1p(features['precip_inches'])
features['ceiling_log1p'] = np.log1p(features['ceiling_ft'])
features['visibility_inverse'] = 1/(1+features['visibility_miles'])

## 8. Save the feature table

Keep the original input lists: A has 26 features; B has 78 for Logistic Regression/KNN,
72 for Random Forest, and 66 for SVM. Current-flight outcomes are excluded from those lists.
Missing inputs will be filled using training data only in notebook 03.

In [54]:
columns = selection['columns']['combined']
category_columns = ['airline', 'origin', 'destination', 'weekday']
numeric_columns = [column for column in columns if column not in category_columns]
numeric_values = features[numeric_columns].to_numpy(dtype=float)
assert not np.isinf(numeric_values).any()
features.to_parquet(OUT / 'features.parquet', index=False)
split_summary = features.groupby('split')['flight_date'].agg(['size', 'min', 'max'])
display(split_summary)

,size,min,max
split,,,
test,10000,2026-07-01,2026-07-31
train,40000,2025-01-01,2025-12-31


Weather availability and the seven-day history gap are assumptions of this study.
Continue to [03 — Model Training](03_model_training.ipynb).